In [1]:
import os
import anthropic

# Set your API key: 
os.environ["ANTHROPIC_API_KEY"] = "your api key here"
client = anthropic.Anthropic()

In [2]:
def step_parse(user_request: str) -> str:
    r = client.messages.create(
        model="claude-sonnet-4-5", max_tokens=300,
        system="Extract travel details. Return JSON only, no markdown.",
        messages=[{"role": "user", "content":
            f"Extract: destination, duration_days, travel_style, budget_level from: {user_request}"}])
    return r.content[0].text

In [3]:
def step_research(parsed: str) -> str:
    r = client.messages.create(
        model="claude-sonnet-4-5", max_tokens=500,
        system="You are a travel researcher. Return JSON list of top 5 must-do experiences only.",
        messages=[{"role": "user", "content":
            f"Based on this trip profile: {parsed}\nList top 5 experiences with name, duration_hours, cost_level."}])
    return r.content[0].text

In [4]:
def step_schedule(parsed: str, experiences: str) -> str:
    r = client.messages.create(
        model="claude-sonnet-4-5", max_tokens=600,
        system="You are a travel planner. Create a logical day-by-day schedule.",
        messages=[{"role": "user", "content":
            f"Trip profile: {parsed}\nAvailable experiences: {experiences}\nCreate a day-by-day schedule as JSON."}])
    return r.content[0].text

In [5]:
def step_write(schedule: str) -> str:
    r = client.messages.create(
        model="claude-sonnet-4-5", max_tokens=800,
        system="You are a travel writer. Write an engaging, practical itinerary.",
        messages=[{"role": "user", "content":
            f"Convert this schedule into a friendly, detailed travel itinerary:\n{schedule}"}])
    return r.content[0].text


In [6]:
request = "5 days in Kyoto, Japan. I love temples and food. Mid-range budget."


In [7]:
parsed      = step_parse(request)
experiences = step_research(parsed)
schedule    = step_schedule(parsed, experiences)
itinerary   = step_write(schedule)

print("=== PARSED ===");      print(parsed)
print("\n=== EXPERIENCES ==="); print(experiences)
print("\n=== SCHEDULE ===");    print(schedule)
print("\n=== FINAL ===");       print(itinerary)

=== PARSED ===
```json
{
  "destination": "Kyoto, Japan",
  "duration_days": 5,
  "travel_style": "temples and food",
  "budget_level": "mid-range"
}
```

=== EXPERIENCES ===
```json
[
  {
    "name": "Fushimi Inari Shrine Hike",
    "duration_hours": 3,
    "cost_level": "free",
    "description": "Walk through thousands of iconic vermillion torii gates winding up Mount Inari. Best visited early morning to avoid crowds."
  },
  {
    "name": "Arashiyama Bamboo Grove & Temple Circuit",
    "duration_hours": 4,
    "cost_level": "low",
    "description": "Explore the enchanting bamboo forest and visit Tenryu-ji Temple. Combine with nearby Okochi Sanso Villa for panoramic views."
  },
  {
    "name": "Nishiki Market Food Tour",
    "duration_hours": 2.5,
    "cost_level": "mid-range",
    "description": "Sample Kyoto specialties at this 400-year-old market: fresh sashimi, tsukemono pickles, tamagoyaki, and matcha treats."
  },
  {
    "name": "Kinkaku-ji (Golden Pavilion) & Ryoan-ji Rock